[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a> [View on GitHub](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb)

# Test and Grade the Agent

Turn the failures from notebooks 1–2 into an 11-question golden suite with ground truth
computed from the same seed as the dataset.

Four families of checks, mirroring real production eval stacks:

| family | checks | catches |
|---|---|---|
| **Execution** | generated code parses (AST), runs without traceback | syntax garbage, retry storms, hallucinated columns |
| **Router** | the model actually called the tool this case requires | mis-routing, skipping the schema check, "answering from vibes" |
| **Ground truth** | `abs(agent - expected) < 0.01` on the FINAL ANSWER number | sign inversion, fragmentation, wrong filters |
| **LLM-as-a-judge** | 1-5 rubric on actionability / heuristic adherence / numerical fidelity | plausible-sounding nonsense, fabricated balances |

The checks fail closed: missing output or a crashed tool means FAIL.


---
## Bootstrap

Same as notebook 2, plus the golden suite - loaded here because this is where
evaluation starts.


In [ ]:
import urllib.request
import os

if not os.path.exists("utils.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils

utils.load_data()
utils.load_golden()
utils.connect()

from utils import (CREATIVE_SYSTEM_PROMPT, HARDENED_SYSTEM_PROMPT, JUDGE_MODEL, GOLDEN,
                   MAX_TOKENS, client, df, run_agent, waterfall)

print("bootstrap done |", len(GOLDEN["cases"]), "golden cases loaded")


---
## The harness

Each check reads a run result. Numeric answers use the `FINAL ANSWER: <number>` convention
so they can be compared with ground truth.


In [ ]:
import re

FINAL_ANSWER_RE = re.compile(r"final\s+answer\s*:\s*\$?\s*(-?[\d,]+(?:\.\d+)?)", re.IGNORECASE)


def extract_final_number(text):
    """Return the numeric value after FINAL ANSWER, or None if missing."""
    m = FINAL_ANSWER_RE.search(text or "")
    if not m:
        return None
    try:
        return float(m.group(1).replace(",", ""))
    except ValueError:
        return None


### Check the router

This function checks whether the agent called the tool required for each case.


In [ ]:
REQUIRED_TOOLS = {
    "total_expenses_october": ["analyze_transactions"],
    "net_expenses_october_after_refunds": ["analyze_transactions"],
    "starbucks_total": ["analyze_transactions"],
    "merchant_location": ["fetch_data_schema"],
    "total_income": ["analyze_transactions"],
    "top_spending_category": ["analyze_transactions"],
    "avg_monthly_spending": ["analyze_transactions"],
    "largest_single_expense": ["analyze_transactions"],
    "transaction_count_may": ["analyze_transactions"],
    "advisory_budget_5030_20": ["financial_rules_lookup"],
    "advisory_emergency_fund": ["financial_rules_lookup"],
}


def check_router(case, result):
    """Router eval: did the model actually route to the tool this case requires?"""
    required = REQUIRED_TOOLS.get(case["id"], [])
    used = []
    for m in result.get("messages", []):
        if m.get("role") == "assistant":
            for tc in m.get("tool_calls") or []:
                used.append((tc.get("function") or {}).get("name"))
    return {"tools_used": ", ".join(dict.fromkeys(used)), "router_ok": all(t in used for t in required)}


### Check code execution

Each code call should parse and run successfully.


In [ ]:
import ast


def check_code(result):
    """Check that every generated code chunk parses and runs successfully."""
    chunks = result["executed_code"]
    code_valid = bool(chunks)
    for chunk in chunks:
        try:
            ast.parse(chunk["code"])
        except SyntaxError:
            code_valid = False

    return {
        "n_chunks": len(chunks),
        "code_valid": code_valid,
        "no_traceback": bool(chunks) and all(chunk["ok"] for chunk in chunks),
    }


### Check merchant names

Reduce a merchant name to a word we can look for in the answer.


In [ ]:
def _merchant_token(merchant):
    """Get a distinctive word to check that the answer names the merchant."""
    tokens = [t for t in re.split(r"[^A-Za-z]+", merchant.upper()) if len(t) >= 4]
    return tokens[0] if tokens else merchant.upper()[:6]


### Check numeric answers

Compare the extracted final number with the computed ground truth.


In [ ]:
def check_numeric(case, result):
    """Check the final number against the case's expected value."""
    expected = case["expected"]
    got = extract_final_number(result["answer"])
    out = {"extracted": got, "numeric_ok": False}
    if got is None:
        return out
    out["numeric_ok"] = abs(got - float(expected["value"])) <= case["tolerance"] + 1e-9
    if "category" in expected:
        out["category_ok"] = expected["category"].lower() in result["answer"].lower()
    if "merchant_contains" in expected:
        out["merchant_ok"] = _merchant_token(expected["merchant_contains"]).lower() in result["answer"].lower()
    return out


### Decide whether a case passes

Require every check for that case type; missing checks do not pass.


In [ ]:
def _case_passes(case, checks):
    """Pass only when all checks required for this case type pass."""
    ctype = case["type"]
    if ctype == "numeric":
        required = ("answered", "code_valid", "no_traceback", "numeric_ok", "router_ok")
        return all(checks.get(key) for key in required) and all(
            checks.get(key, True) for key in ("category_ok", "merchant_ok")
        )
    if ctype == "behavior":
        return all(checks.get(key) for key in ("answered", "code_valid", "no_traceback", "router_ok", "judge_pass"))
    return all(checks.get(key) for key in ("answered", "router_ok", "judge_pass"))


### Prepare the judge prompt

The judge compares advisory answers with the same case ground truth.


In [ ]:
JUDGE_SYSTEM_PROMPT = """You are a strict, neutral evaluation judge grading an AI financial assistant. Compare the answer against the provided ground truth and return JSON only."""


Parse the judge response as JSON. If it is not valid JSON, the case fails.


In [ ]:
import json


def _parse_json(text):
    """Parse the judge's JSON response; malformed output fails closed."""
    try:
        return json.loads(text)
    except Exception:
        return {"pass": False, "reason": "judge output unparseable", "scores": {}}


Send the answer, expected result, and rubric to the judge.


In [ ]:
def judge_case(case, result):
    """Grade an answer against the case's ground truth and rubric."""
    payload = {
        "user_query": case["query"],
        "agent_answer": result["answer"],
        "ground_truth": case["expected"],
        "grading_rubric": case.get("judge_rubric"),
        "instructions": "Score each rubric dimension from 1 to 5. Pass only if every score is at least 4 and no figure contradicts the ground truth. Return JSON with scores, pass, and reason.",
    }
    response = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                  {"role": "user", "content": json.dumps(payload)}],
        temperature=0.0,
        max_tokens=MAX_TOKENS,
    )
    return _parse_json(response.choices[0].message.content)


`run_suite` runs every golden case and combines its checks into one scorecard.


In [ ]:
import time

import pandas as pd

SUITE_RESULTS = {}


def run_suite(label, system_prompt, temperature, agent_fn=None, judge_fn=None):
    """Run all golden cases into a scorecard. agent_fn/judge_fn are injectable
    so the CI mock twin can drive the exact same harness deterministically."""
    agent_fn = agent_fn or run_agent
    judge_fn = judge_fn or judge_case
    rows = []
    SUITE_RESULTS[label] = {}
    for case in GOLDEN["cases"]:
        t0 = time.time()
        result = agent_fn(case["query"], system_prompt=system_prompt, temperature=temperature, label=f"{label}/{case['id']}")
        SUITE_RESULTS[label][case["id"]] = result
        checks = {"answered": bool(result["answer"].strip())}
        checks.update(check_router(case, result))
        if case["type"] in ("numeric", "behavior"):
            checks.update(check_code(result))
        if case["type"] == "numeric":
            checks.update(check_numeric(case, result))
        if case["type"] in ("advisory", "behavior"):
            verdict = judge_fn(case, result)
            checks["judge_pass"] = bool(verdict.get("pass"))
            checks["judge_reason"] = str(verdict.get("reason", ""))[:70]
        checks["PASS"] = _case_passes(case, checks)
        checks["latency_s"] = round(time.time() - t0, 2)
        row = {"case": case["id"], "type": case["type"], "trap": case["trap"] or ""}
        row.update({k: v for k, v in checks.items() if k != "judge_reason"})
        row["answer_preview"] = result["answer"].replace("\n", " ")[:110]
        rows.append(row)
    scorecard = pd.DataFrame(rows)
    print(f"### {label}: {int(scorecard['PASS'].sum())}/{len(scorecard)} cases pass ({scorecard['PASS'].mean():.0%})")
    return scorecard


### Step 1 - Baseline: the hardened agent

Same tools, same model - but now the system prompt encodes the lessons from notebook 1
(schema-first, signed amounts, fuzzy merchant matching, `FINAL ANSWER` discipline). Run
the suite:


In [ ]:
baseline_scorecard = run_suite("baseline", HARDENED_SYSTEM_PROMPT, 0.0)
cols = ["case", "type", "trap", "router_ok", "tools_used", "answered", "code_valid",
        "no_traceback", "numeric_ok", "extracted", "judge_pass", "PASS", "latency_s"]
baseline_scorecard[[c for c in cols if c in baseline_scorecard.columns]]


In [ ]:
baseline_failed = [row["case"] for row in baseline_scorecard.to_dict(orient="records") if not row["PASS"]]
print("baseline failures:", baseline_failed or "none")
for cid in baseline_failed:
    res = SUITE_RESULTS["baseline"][cid]
    print(f"\n--- {cid} ---")
    print(res["answer"][:400])
    waterfall(res, max_events=12)


### Step 2 - The regression injection

A stakeholder drops by: *"the agent feels robotic - make it more **creative and
conversational**, loosen it up a bit."* Sounds harmless. It is the most common way eval
suites start earning their keep. Apply the "improvement": a friendlier persona, freedom
to estimate and round, and `temperature=0.7` for extra sparkle.


In [ ]:
regression_scorecard = run_suite("regression", CREATIVE_SYSTEM_PROMPT, 0.7)
regression_scorecard[[c for c in cols if c in regression_scorecard.columns]]


### Step 3 - The diff: what broke, and how the suite caught it


In [ ]:
merged = baseline_scorecard[["case", "PASS"]].merge(
    regression_scorecard[["case", "PASS"]], on="case", suffixes=("_baseline", "_creative")
)
merged["regressed?"] = merged["PASS_baseline"] & ~merged["PASS_creative"]

b = int(baseline_scorecard["PASS"].sum())
r = int(regression_scorecard["PASS"].sum())
total_cases = len(GOLDEN["cases"])
print(f"baseline   : {b}/{total_cases}")
print(f"regression : {r}/{total_cases}")
print(f"verdict    : REGRESSION DETECTED - {int(merged['regressed?'].sum())} case(s) broke" if r < b else "no regression detected")
merged


In [ ]:
regressed_cases = merged.loc[merged["regressed?"], "case"].tolist()
if regressed_cases:
    cid = regressed_cases[0]
    res = SUITE_RESULTS["regression"][cid]
    print(f"=== autopsy of the first regressed case: {cid} ===\n")
    print("agent answer:\n", res["answer"][:400])
    print("\nexecuted code:")
    for c in res["executed_code"]:
        print(" ", c["code"][:120].replace("\n", " | "), "-> ok:", c["ok"])
    waterfall(res, max_events=15)
else:
    print("nothing regressed - the creative prompt happened to survive. Re-run: stochastic regressions are exactly why we automate this.")


### The router column, specifically

Look at `router_ok` / `tools_used` above: the regression didn't just make answers wrong -
it changed **routing behavior** (skipping `fetch_data_schema`, answering advisory
questions without consulting `financial_rules_lookup`). Routing is a capability with its
own check because "the model picked the wrong tool" is a distinct production failure
from "the model computed the wrong number".

### Your turn: beat the baseline (optional, costs ~11 live calls)

The suite is not just a gate - it is a *workbench*. Copy the hardened prompt, change it,
and see whether you can reach 11/11 without losing the conversational quality the
stakeholder wanted.


In [ ]:
MY_SYSTEM_PROMPT = HARDENED_SYSTEM_PROMPT + """

8. When delivering bad news (overspending, insufficient savings), lead with the concrete number,
   then one actionable step. Warm tone is fine; vague reassurance is not."""

mine_scorecard = run_suite("mine", MY_SYSTEM_PROMPT, 0.0)
print(f"baseline: {int(baseline_scorecard['PASS'].sum())}/{len(GOLDEN['cases'])}   yours: {int(mine_scorecard['PASS'].sum())}/{len(GOLDEN['cases'])}")


---
## Optional - paint your eval results onto the traces

If you ran notebook 2's Phoenix setup **in this same runtime**, the suite you just
executed was traced: every case is a CHAIN span named `baseline/<case>`. This cell
attaches the pass/fail verdict to those spans - the pattern for annotating production
traces with eval scores:


In [ ]:
if utils.SPAN_TRACER is None:
    print("No tracer attached (run notebook 2's Phoenix cells in this runtime first) - skipping.")
else:
    import pandas as _pd
    from phoenix.client import Client

    spans = Client().spans.get_spans(project_identifier=f"workshop-{utils.seat_tag()}", limit=2000)
    by_name = {s["name"]: (s.get("context") or {}).get("span_id") for s in spans if s.get("name")}

    ann_rows = []
    for row in baseline_scorecard.to_dict(orient="records"):
        span_id = by_name.get(f"baseline/{row['case']}")
        if span_id:
            ann_rows.append({"span_id": span_id, "label": "PASS" if row["PASS"] else "FAIL",
                             "score": float(bool(row["PASS"])), "explanation": row.get("answer_preview", "")[:100]})
    if ann_rows:
        Client().spans.log_span_annotations_dataframe(
            dataframe=_pd.DataFrame(ann_rows), annotation_name="Golden Suite", annotator_kind="CODE",
        )
        print(f"annotated {len(ann_rows)} spans - open the Phoenix UI and click a trace to see the verdict")
    else:
        print("no matching spans found yet - re-run the baseline suite and try again")


---
## Production Governance & CI/CD

The same suite, wired into GitHub Actions (`.github/workflows/evals.yml` in the repo):

```yaml
name: evals
on: [push, pull_request]
jobs:
  golden-suite:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install -r requirements.txt
      - run: python scripts/generate_dataset.py --check
      - run: python scripts/run_evals.py --mode mock --config baseline --min-pass-rate 0.9
      - run: python scripts/run_evals.py --mode mock --config regression --expect-regression
```

What each line defends:
- `generate_dataset.py --check` - **data drift**: the CSV and frozen ground truths must stay consistent with the seed
- `run_evals.py --config baseline` - **capability regression**: the agent must clear the golden suite
- `--expect-regression` - **the gate itself is tested**: the harness must catch a known-bad configuration (a deterministic mock run, zero API cost)

`scripts/run_evals.py` drives the *same* `utils.py` harness these notebooks use, with a
deterministic mock agent injected - notebook and CI can never drift apart. Push/PR checks
cost nothing; the scheduled live check runs through the workshop gateway with a
`LITELLM_API_KEY` GitHub secret.

**Golden dataset hygiene, briefly:** regenerate ground truth from code, never edit JSON
by hand; version the seed; add a case the moment a new failure mode reaches production;
delete cases that no longer map to a real capability.

**Discussion starters for Q&A:**
- Which of your current LLM features have *any* automated eval today?
- What would your first 10 golden cases be - and who signs off on the ground truth?
- Where does the eval gate belong: pre-merge, nightly, or both?

*Thank you - and may all your regressions be caught by robots, not users.*


---
### Workshop home

[Return to the workshop README](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/README.md)
